L'implémentation d'une fenêtre glissante (Sliding Window) consiste à restreindre l'historique injecté dans le prompt aux seuls messages les plus récents afin d'éviter la saturation de la fenêtre de contexte du modèle et de réduire la latence de l'inférence
## 1. Structure des messages et du Prompt
La mémoire dans LangChain repose sur l'alternance d'objets HumanMessage et AIMessage injectés dans un placeholder au sein du prompt

In [ ]:
import os
from langchain_ollama import ChatOllama  
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
 

OLLAMA_HOST = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
llm = ChatOllama(model="gemma3:4b", base_url=OLLAMA_HOST, temperature=0) # Ou tout modèle supportant cette fonction

# Définition du prompt avec un placeholder pour l'historique
prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un assistant support technique local."),
    MessagesPlaceholder(variable_name="historique"),
    ("human", "{input}")
])

Pour limiter l'historique aux 5 derniers messages, vous devez manipuler la liste des messages avant de l'envoyer au modèle. En Python, cela se fait par un simple découpage de liste (slicing)

In [ ]:
# Simulation d'un historique stocké (par exemple dans st.session_state)
# [7] recommande de maintenir une liste d'objets HumanMessage/AIMessage
historique_complet = [
    HumanMessage(content="Mon PC est lent"),
    AIMessage(content="Avez-vous redémarré ?"),
    AIMessage(content="Avez-vous essayé le ctr-alt-delete ?"),
    AIMessage(content="Êtes vous branché sur le secteur ?"),
    AIMessage(content="Avez-vous récemment installé une nouvelle application ?"),
]

# Limitation aux 5 derniers messages (Fenêtre glissante)
# Cette opération garantit que seuls les 5 échanges les plus récents 
# sont conservés pour l'appel au LLM [1, 2].
historique_limite = historique_complet[-5:]

# Invocation de la chaîne avec l'historique tronqué
chaine = prompt | llm
resultat= chaine.invoke({"input": "Oui, c'est fait.", "historique": historique_limite})

print(resultat)